In [ ]:
from pathlib import Path as _ProjectPath
import sys as _project_sys

_project_root = next(
    (
        p
        for p in (_ProjectPath.cwd(), *_ProjectPath.cwd().parents)
        if (p / "project_config.json").is_file()
    ),
    None,
)
if _project_root is None:
    raise FileNotFoundError(
        "Open VS Code in the organized project root before running this notebook."
    )
if str(_project_root / "tools") not in _project_sys.path:
    _project_sys.path.insert(0, str(_project_root / "tools"))
from project_paths import ProjectPaths as _ProjectPaths

_paths = _ProjectPaths(_project_root, 'rmhd_checkpoint')

#!pip install pandas numpy scikit-learn matplotlib seaborn shap lime wordcloud


import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import shap
import lime
from lime.lime_text import LimeTextExplainer


sns.set(style="whitegrid", palette="muted", font_scale=1.2)
plt.rcParams["figure.figsize"] = (10, 5)


In [ ]:
data_path = _paths.rmhd_dir


df = pd.read_csv(
    _paths.input_path(os.path.join(data_path, "adhd_2019_features_tfidf_256.csv"))
)


print(df.head())
print(df.info())


print(df.isnull().sum())


In [ ]:
pip install wordcloud


In [ ]:
print(df.columns)


In [ ]:
df = df.dropna(subset=['post', 'subreddit'])
df['post'] = df['post'].astype(str).str.strip()

print("After cleaning:", df.shape)
print(df[['subreddit', 'post']].head())


In [ ]:
import glob

adhd_files = _paths.original_root_csv_paths()
print("Found files:", adhd_files)


In [ ]:
import pandas as pd
import glob
import os


path = _paths.rmhd_dir


adhd_files = glob.glob(os.path.join(path, "adhd*.csv"))
print("ADHD files found:", adhd_files)


df_list = []
for file in adhd_files:
    try:
        temp = pd.read_csv(_paths.input_path(file))
        temp["source_file"] = os.path.basename(file)
        df_list.append(temp)
    except Exception as e:
        print(f"⚠️ Error reading {file}: {e}")

if df_list:
    adhd_df = pd.concat(df_list, ignore_index=True)
    print("Final ADHD dataframe shape:", adhd_df.shape)
    print("Columns:", adhd_df.columns.tolist()[:15], "...")
else:
    print("⚠️ No ADHD CSVs were loaded")


In [ ]:
import os
import glob
import pandas as pd


data_path = _paths.rmhd_dir


all_files = glob.glob(os.path.join(data_path, "*.csv"))
print("Total CSV files found:", len(all_files))

df_list = []

for file in all_files:
    try:

        disorder = os.path.basename(file).split("_")[0].lower()

        temp = pd.read_csv(_paths.input_path(file))
        temp["source_file"] = os.path.basename(file)
        temp["disorder"] = disorder
        df_list.append(temp)

    except Exception as e:
        print(f"⚠️ Error reading {file}: {e}")


if df_list:
    full_df = pd.concat(df_list, ignore_index=True)
    print("Final merged dataframe shape:", full_df.shape)
    print("Disorders present:", full_df["disorder"].unique())
else:
    print("⚠️ No CSVs merged. Check file path and naming pattern.")


In [ ]:
relevant_disorders = [
    "adhd",
    "autism",
    "addiction",
    "alcoholism",
    "anxiety",
    "healthanxiety",
    "socialanxiety",
    "bipolarreddit",
    "bpd",
    "depression",
    "ptsd",
    "schizophrenia",
    "suicidewatch",
    "lonely",
    "mentalhealth",
]

filtered_df = full_df[full_df["disorder"].isin(relevant_disorders)].reset_index(
    drop=True
)

print("Filtered dataset shape:", filtered_df.shape)
print("Disorders kept:", filtered_df["disorder"].unique())


In [ ]:
import pandas as pd


df = pd.read_csv(_paths.input_path("merged_dataset.csv"))


relevant_disorders = [
    "addiction",
    "adhd",
    "alcoholism",
    "anxiety",
    "autism",
    "bipolarreddit",
    "bpd",
    "depression",
    "healthanxiety",
    "lonely",
    "mentalhealth",
    "ptsd",
    "schizophrenia",
    "socialanxiety",
    "suicidewatch",
]


df_filtered = df[df["label"].isin(relevant_disorders)].copy()


df_filtered.reset_index(drop=True, inplace=True)


df_filtered.to_csv(_paths.output_path("filtered_dataset.csv"), index=False)

print("✅ Filtering complete. New shape:", df_filtered.shape)
print("✅ Unique labels:", df_filtered['label'].unique())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv(_paths.input_path("filtered_dataset.csv"))


print("Dataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum().sort_values(ascending=False).head(10))


print("\nDisorder Categories:", df['label'].unique())


plt.figure(figsize=(12, 6))
sns.countplot(
    data=df, x="label", order=df["label"].value_counts().index, palette="viridis"
)
plt.xticks(rotation=45, ha="right")
plt.title("Class Distribution of Disorders")
plt.show()


df['post_length'] = df['post'].astype(str).apply(len)

plt.figure(figsize=(10, 5))
sns.histplot(df['post_length'], bins=50, kde=True, color="teal")
plt.title("Distribution of Post Lengths (characters)")
plt.xlabel("Post Length")
plt.ylabel("Frequency")
plt.show()


if 'sent_compound' in df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(df['sent_compound'], bins=50, kde=True, color="orange")
    plt.title("Distribution of Sentiment Compound Scores")
    plt.xlabel("Sentiment Score (-1 = Negative, +1 = Positive)")
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x='label', y='sent_compound', palette="coolwarm")
    plt.xticks(rotation=45)
    plt.title("Sentiment Distribution by Disorder Label")
    plt.xlabel("Disorder (Label)")
    plt.ylabel("Sentiment Compound Score")
    plt.show()


readability_cols = [
    'automated_readability_index',
    'coleman_liau_index',
    'flesch_kincaid_grade_level',
    'flesch_reading_ease',
    'gunning_fog_index',
    'smog_index',
]

available_readability = [col for col in readability_cols if col in df.columns]

if available_readability:
    plt.figure(figsize=(12, 6))
    df[available_readability].mean().sort_values().plot(kind="bar", color="slateblue")
    plt.title("Average Readability Scores Across Posts")
    plt.ylabel("Mean Score")
    plt.show()


num_cols = df.select_dtypes(include=['float64', 'int64']).columns
corr = df[num_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()


df['word_count'] = df['post'].astype(str).apply(lambda x: len(x.split()))

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='label', y='word_count', palette="Set2")
plt.xticks(rotation=45)
plt.title("Word Count Distribution by Disorder Label")
plt.xlabel("Disorder (Label)")
plt.ylabel("Word Count")
plt.show()


In [ ]:
import pandas as pd


df = pd.read_csv(_paths.input_path("filtered_dataset.csv"))
print("Dataset shape:", df.shape)
print("Class distribution:\n", df['label'].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['post'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print("TF-IDF vectorization done. Feature shape:", X_train_vec.shape)


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import time


def train_and_evaluate(model, name):
    start = time.time()
    model.fit(X_train_vec, y_train_enc)
    y_pred = model.predict(X_test_vec)
    end = time.time()
    print(f"\n{name} Report (Time: {end-start:.2f}s):")
    print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
    print(f"{name} Accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")
    return end - start


In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, n_jobs=-1)
train_and_evaluate(log_reg, "Logistic Regression")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

log_reg = LogisticRegression(max_iter=1000, n_jobs=-1)


start = time.time()
log_reg.fit(X_train_vec, y_train_enc)
y_pred = log_reg.predict(X_test_vec)
end = time.time()

print(f"\nLogistic Regression Report (Time: {end-start:.2f}s):")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
print(f"Accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")


cm = confusion_matrix(y_test_enc, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
)
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC(max_iter=5000)

start = time.time()
svm.fit(X_train_vec, y_train_enc)
y_pred = svm.predict(X_test_vec)
end = time.time()

print(f"\nLinear SVM Report (Time: {end-start:.2f}s):")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
print(f"Accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")

cm = confusion_matrix(y_test_enc, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
)
plt.title("Linear SVM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

start = time.time()
rf.fit(X_train_vec, y_train_enc)
y_pred = rf.predict(X_test_vec)
end = time.time()

print(f"\nRandom Forest Report (Time: {end-start:.2f}s):")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
print(f"Accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")

cm = confusion_matrix(y_test_enc, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Oranges',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
)
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
    class_weight='balanced',
    max_features='sqrt',
)
train_evaluate(rf, X_train_vec, X_test_vec, "Random Forest")


In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC


In [ ]:
df = pd.read_csv(_paths.input_path("filtered_dataset.csv"))
print("Dataset shape:", df.shape)
print("Class distribution:\n", df['label'].value_counts())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['post'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=15000, stop_words='english', ngram_range=(1, 2), min_df=5, max_df=0.95
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print("TF-IDF vectorization done. Feature shape:", X_train_vec.shape)


In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


In [ ]:
def train_evaluate(model, X_train_input, X_test_input, model_name="Model"):
    start = time.time()
    model.fit(X_train_input, y_train_enc)
    y_pred = model.predict(X_test_input)
    end = time.time()

    acc = accuracy_score(y_test_enc, y_pred)
    print(f"\n{model_name} Report (Time: {end-start:.2f}s):")
    print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
    print(f"{model_name} Accuracy: {acc:.4f}")

    cm = confusion_matrix(y_test_enc, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=le.classes_,
        yticklabels=le.classes_,
    )
    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    return acc


In [ ]:
log_reg = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight='balanced', C=1.0)
acc_lr = train_evaluate(log_reg, X_train_vec, X_test_vec, "Logistic Regression")


In [ ]:
svm = LinearSVC(C=0.1, max_iter=5000, class_weight='balanced')
acc_svm = train_evaluate(svm, X_train_vec, X_test_vec, "Linear SVM")


In [ ]:
nb = MultinomialNB()
train_evaluate(nb, X_train_vec, X_test_vec, "Naive Bayes")
